# Hugging Face Applications — Lesson 7: Zero-Shot Classification

> Learning material for **Hugging Face Applications**. Companion to the lesson script `07_Zero_Shot_Classification.py` (same content, runnable without Jupyter).

**Task ID:** HF-207  |  **Folder:** `documentation`


## What is zero-shot classification?

Normal classifiers are stuck with the categories they were **trained** on. **Zero-shot** classification lets you define the categories at inference time — the model has *never seen them*:

> 💬 *"The striker scored in the final minute"*
>
> 🏷️ You provide: `sports`, `finance`, `health`
>
> 🎯 Model: `sports (0.41)`, `health (0.37)`, `finance (0.22)`

"Zero-shot" = **zero examples** of your classes were used in training.

## The trick: Natural Language Inference (NLI)

Behind the scenes, an NLI model answers, for every candidate label:

> *Does the text **entail** (logically imply) this label?*

The label with the strongest "entailment" wins. That's why the task works with labels it never saw — it reasons about the *meaning* of your words.

The pipeline hides all this:


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="typeform/distilbert-base-uncased-mnli",   # an NLI model
    device=-1,
)
# first call downloads the model (~270 MB)


**Classify with your own labels:**

In [ ]:
text = (
    "The central bank raised interest rates again today, citing stubborn "
    "inflation, and markets reacted with a sharp sell-off."
)
labels = ["finance", "sports", "health", "technology"]

result = classifier(text, candidate_labels=labels)

print("Text:", text)
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<12} {score:.3f}")


Note the output shape: `result["labels"]` and `result["scores"]`
— two parallel lists, already sorted best-first.

## Multi-label mode

By default scores sum to 1 (single label). With `multi_label=True` each
label is scored independently — several can be high at once:


In [ ]:
text = "The new fitness tracker measures your heart rate and sleep quality."
labels = ["health tech", "sports", "movies"]

result = classifier(text, candidate_labels=labels, multi_label=True)
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<12} {score:.3f}")


## When to use zero-shot instead of fine-tuning?

| Situation | Choice | 
|-----------|--------|
| few example texts, categories change often | **zero-shot** | 
| fixed categories, need top accuracy | fine-tune (HF-208) |

## Try it yourself

1. Classify a news headline of yours with labels: `politics, sports, weather, crime`.
2. Use the same text with very different label sets — watch scores shift.
3. Swap to `facebook/bart-large-mnli` — stronger but ~1.6 GB.

## Summary

- Zero-shot = labels come from you, at inference time.
- Mechanism: NLI "entailment" scoring per label.
- `multi_label=True` for independent scoring.

**Next lesson:** HF-208 — Fine-Tuning Basics.  |  Extra reading: `../resources/reference_links.md`
